# Week 2 Exercise — Technical Q&A Assistant

Building on the Week 1 technical question answerer, this notebook adds:
- **Gradio UI** with `gr.Blocks` for a polished chat experience
- **Streaming** responses for real-time feedback
- **Expert system prompt** for deep technical explanations
- **Model switching** between OpenAI GPT and local Ollama (Llama)
- **Tool calling** — a Python expression evaluator so the assistant can verify computations
- **Audio input** (speech-to-text via Whisper) and **audio output** (text-to-speech via OpenAI TTS)

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set — GPT models will not work")

In [ ]:
MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

openai_client = OpenAI()
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

clients = {
    MODEL_GPT: openai_client,
    MODEL_LLAMA: ollama_client,
}

## System Prompt

The system prompt gives the assistant deep technical expertise and sets the tone for answers.

In [ ]:
system_prompt = """
You are a senior software engineer and computer science educator with deep expertise in \
Python, JavaScript, algorithms, data structures, system design, and DevOps.

When a user asks a technical question:
1. Provide a clear, concise explanation first.
2. Follow up with a practical code example when relevant.
3. Point out common pitfalls or edge cases.
4. If the question involves a computation or expression you can verify, use the \
   evaluate_python_expression tool to confirm the result before answering.

Respond in well-structured markdown. Be accurate — if you are unsure, say so.
"""

## Tool Definition — Python Expression Evaluator

This tool lets the LLM evaluate simple Python expressions to verify computations \
(e.g. checking the output of a list comprehension, math, or string operation).

In [ ]:
def evaluate_python_expression(expression: str) -> str:
    """Safely evaluate a Python expression and return the result as a string."""
    print(f"TOOL CALLED: evaluate_python_expression({expression!r})")
    ALLOWED_BUILTINS = {
        "abs": abs, "len": len, "max": max, "min": min, "sum": sum,
        "sorted": sorted, "list": list, "dict": dict, "set": set,
        "tuple": tuple, "range": range, "enumerate": enumerate,
        "zip": zip, "map": map, "filter": filter, "round": round,
        "int": int, "float": float, "str": str, "bool": bool,
        "True": True, "False": False, "None": None,
    }
    try:
        result = eval(expression, {"__builtins__": ALLOWED_BUILTINS})
        return json.dumps({"result": repr(result)})
    except Exception as e:
        return json.dumps({"error": str(e)})

In [ ]:
eval_function_spec = {
    "name": "evaluate_python_expression",
    "description": (
        "Evaluate a simple Python expression and return its result. "
        "Use this to verify computations, list comprehensions, math, "
        "or string operations when answering technical questions."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "A valid Python expression to evaluate, e.g. 'sorted([3,1,2])'",
            }
        },
        "required": ["expression"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": eval_function_spec}]

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "evaluate_python_expression":
            args = json.loads(tool_call.function.arguments)
            result = evaluate_python_expression(args["expression"])
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id,
            })
    return responses

## Chat Function with Streaming and Tool Support

In [ ]:
def chat(message, history, model_name):
    client = clients[model_name]
    use_tools = model_name == MODEL_GPT

    history_msgs = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_prompt}]
        + history_msgs
        + [{"role": "user", "content": message}]
    )

    # --- Tool-call loop (non-streaming first pass) ---
    if use_tools:
        response = client.chat.completions.create(
            model=model_name, messages=messages, tools=tools
        )
        while response.choices[0].finish_reason == "tool_calls":
            assistant_msg = response.choices[0].message
            tool_responses = handle_tool_calls(assistant_msg)
            messages.append(assistant_msg)
            messages.extend(tool_responses)
            response = client.chat.completions.create(
                model=model_name, messages=messages, tools=tools
            )

    # --- Streaming final answer ---
    stream = client.chat.completions.create(
        model=model_name, messages=messages, stream=True
    )
    response_text = ""
    for chunk in stream:
        response_text += chunk.choices[0].delta.content or ""
        yield response_text

## Audio Support

- **Input**: Record audio in the browser → transcribe with Whisper → feed text to the chat
- **Output**: The assistant's reply is spoken back via OpenAI TTS

In [ ]:
def transcribe_audio(audio_path):
    """Convert recorded audio to text using Whisper."""
    if audio_path is None:
        return ""
    with open(audio_path, "rb") as f:
        transcript = openai_client.audio.transcriptions.create(
            model="whisper-1", file=f
        )
    return transcript.text


def text_to_speech(text):
    """Generate speech audio bytes from text using OpenAI TTS."""
    if not text:
        return None
    response = openai_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=text,
    )
    return response.content

## Gradio UI

A `gr.Blocks` interface that wires together:
- A chatbot panel with streaming
- A model selector dropdown
- Text input + audio input
- Audio output that auto-plays the response

In [ ]:
def add_user_message(user_text, audio, history, model_name):
    """Prepare the user message (from text or audio) and append to history."""
    message = user_text.strip() if user_text else ""
    if not message and audio is not None:
        message = transcribe_audio(audio)
    if not message:
        return "", None, history, history
    history = history + [{"role": "user", "content": message}]
    return "", None, history, history


def bot_respond(history, model_name):
    """Stream the assistant response into the chatbot and produce TTS audio."""
    if not history or history[-1]["role"] != "user":
        yield history, None
        return

    user_message = history[-1]["content"]
    prior_history = history[:-1]

    history = history + [{"role": "assistant", "content": ""}]
    full_reply = ""
    for partial in chat(user_message, prior_history, model_name):
        full_reply = partial
        history[-1]["content"] = partial
        yield history, None

    # Generate TTS for the final reply (GPT models only — Ollama has no TTS)
    audio_bytes = None
    if model_name == MODEL_GPT:
        try:
            audio_bytes = text_to_speech(full_reply)
        except Exception as e:
            print(f"TTS failed: {e}")
    yield history, audio_bytes

In [ ]:
with gr.Blocks(title="Technical Q&A Assistant") as ui:
    gr.Markdown("## Technical Q&A Assistant")
    gr.Markdown(
        "Ask any technical question via text or voice. "
        "Switch models with the dropdown. The assistant can run Python "
        "expressions to verify its answers."
    )

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=[MODEL_GPT, MODEL_LLAMA],
            value=MODEL_GPT,
            label="Model",
            interactive=True,
        )

    chatbot = gr.Chatbot(height=500, type="messages")

    with gr.Row():
        audio_output = gr.Audio(label="Response Audio", autoplay=True)

    with gr.Row():
        text_input = gr.Textbox(
            label="Type your question",
            placeholder="e.g. Explain Python generators...",
            scale=4,
        )
        audio_input = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="Or record audio",
            scale=1,
        )

    with gr.Row():
        clear_audio_btn = gr.Button("Clear Audio")
        clear_chat_btn = gr.Button("Clear Chat")

    clear_audio_btn.click(lambda: None, outputs=[audio_output])
    clear_chat_btn.click(lambda: ([], None), outputs=[chatbot, audio_output])

    # Wire up: user submits text or audio → add message → bot streams response
    text_input.submit(
        add_user_message,
        inputs=[text_input, audio_input, chatbot, model_dropdown],
        outputs=[text_input, audio_input, chatbot, chatbot],
    ).then(
        bot_respond,
        inputs=[chatbot, model_dropdown],
        outputs=[chatbot, audio_output],
    )

    # Also allow submitting via the audio stop event
    audio_input.stop_recording(
        add_user_message,
        inputs=[text_input, audio_input, chatbot, model_dropdown],
        outputs=[text_input, audio_input, chatbot, chatbot],
    ).then(
        bot_respond,
        inputs=[chatbot, model_dropdown],
        outputs=[chatbot, audio_output],
    )

ui.launch(inbrowser=True)